In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [25]:
!kaggle kernels output manvidhamija/ids-18 -p /path/to/dest

clean_chunk_1.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_10.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_11.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_12.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_13.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_14.csv: Skipping, found more recently modified local copy (use --force to force download)
clean_chunk_15.csv: Skipping, found more recently modified local copy (use --force to force download)
^C
User cancelled operation


In [26]:
!kaggle kernels output manvidhamija/ids-18-2 -p /path/to/dest

feature_columns.pkl: Skipping, found more recently modified local copy (use --force to force download)
label_encoder.pkl: Skipping, found more recently modified local copy (use --force to force download)
standard_scaler.pkl: Skipping, found more recently modified local copy (use --force to force download)
best_transformer_model.pth: Skipping, found more recently modified local copy (use --force to force download)
feature_columns.pkl: Skipping, found more recently modified local copy (use --force to force download)
label_encoder.pkl: Skipping, found more recently modified local copy (use --force to force download)
standard_scaler.pkl: Skipping, found more recently modified local copy (use --force to force download)
Kernel log downloaded to /path/to/dest/ids-18-2.log 


In [33]:
# ============================================================
# Standard Library
# ============================================================

import os
import gc
import random
import warnings
import joblib
from collections import defaultdict

# ============================================================
# Data Processing
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

# ============================================================
# Machine Learning
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# PyTorch
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from torch.amp import (
    autocast,
    GradScaler
)

# ============================================================
# Utilities
# ============================================================

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("=" * 60)
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

PyTorch Version : 2.10.0+cu128
CUDA Available  : True
GPU : Tesla T4


In [62]:
# ============================================================
# Cell 2 - Configuration
# ============================================================

# ============================================================
# Data Paths
# ============================================================

DATA_PATHS = {

    "htre_dataset": "/path/to/dest/CIC_IDS2018_HTRE_Final",

    "artifacts": "/path/to/dest/Transformer_Pipeline"

}

INPUT_DIR = DATA_PATHS["htre_dataset"]

# ============================================================
# Artifact Paths
# ============================================================

ARTIFACT_PATHS = {

    "scaler": os.path.join(
        DATA_PATHS["artifacts"],
        "standard_scaler.pkl"
    ),

    "label_encoder": os.path.join(
        DATA_PATHS["artifacts"],
        "label_encoder.pkl"
    ),

    "feature_columns": os.path.join(
        DATA_PATHS["artifacts"],
        "feature_columns.pkl"
    )

}

# ============================================================
# Data Configuration
# ============================================================

DATA_CONFIG = {

    "window_size": 20,

    "batch_size": 256,

    "num_workers": 2

}

# ============================================================
# Model Configuration
# ============================================================

MODEL_CONFIG = {

    "d_model": 128,

    "nhead": 4,

    "num_layers": 2,

    "dim_feedforward": 256,

    "dropout": 0.2,

    "attention_hidden" : 64

}

# ============================================================
# Training Configuration
# ============================================================

TRAIN_CONFIG = {

    "epochs": 15,

    "learning_rate": 1e-3,

    "weight_decay": 1e-4,

    "patience": 5

}

# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# Configuration Summary
# ============================================================

print("=" * 65)
print("Configuration Loaded")
print("=" * 65)

print(f"Dataset Directory : {INPUT_DIR}")
print(f"Artifacts         : {DATA_PATHS['artifacts']}")
print(f"Device            : {DEVICE}")

print()

print("Data Config")
print(DATA_CONFIG)

print()

print("Model Config")
print(MODEL_CONFIG)

print()

print("Training Config")
print(TRAIN_CONFIG)

print("=" * 65)

Configuration Loaded
Dataset Directory : /path/to/dest/CIC_IDS2018_HTRE_Final
Artifacts         : /path/to/dest/Transformer_Pipeline
Device            : cuda

Data Config
{'window_size': 20, 'batch_size': 256, 'num_workers': 2}

Model Config
{'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256, 'dropout': 0.2, 'attention_hidden': 64}

Training Config
{'epochs': 15, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'patience': 5}


In [35]:
# ============================================================
# Cell 3 - Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=SEED):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


set_seed()

print("=" * 65)
print(f"Random Seed : {SEED}")
print("=" * 65)

Random Seed : 42


In [36]:
# ============================================================
# Cell 4 - Load Artifacts
# ============================================================

# Load Standard Scaler
scaler = joblib.load(ARTIFACT_PATHS["scaler"])

# Load Label Encoder
label_encoder = joblib.load(ARTIFACT_PATHS["label_encoder"])

# Load Feature Columns
FEATURE_COLUMNS = joblib.load(ARTIFACT_PATHS["feature_columns"])

# ============================================================
# Dataset Columns
# ============================================================

TIMESTAMP_COLUMN = "Timestamp"
LABEL_COLUMN = "Label"

# ============================================================
# Derived Configuration
# ============================================================

MODEL_CONFIG["input_dim"] = len(FEATURE_COLUMNS)

NUM_CLASSES = len(label_encoder.classes_)
CLASS_NAMES = list(label_encoder.classes_)

# ============================================================
# Summary
# ============================================================

print("=" * 65)
print("Artifacts Loaded Successfully")
print("=" * 65)

print(f"Number of Features : {len(FEATURE_COLUMNS)}")
print(f"Number of Classes  : {NUM_CLASSES}")

print("\nFeature Preview")
print(FEATURE_COLUMNS[:5], "...")

print("\nClasses")

for idx, cls in enumerate(CLASS_NAMES):
    print(f"{idx:2d} : {cls}")

print("=" * 65)

Artifacts Loaded Successfully
Number of Features : 43
Number of Classes  : 14

Feature Preview
['Init Fwd Win Byts', 'Fwd Seg Size Min', 'Fwd Header Len', 'Dst Port', 'Flow Duration'] ...

Classes
 0 : Benign
 1 : Brute Force -Web
 2 : Brute Force -XSS
 3 : DDOS attack-HOIC
 4 : DDOS attack-LOIC-UDP
 5 : DDoS attacks-LOIC-HTTP
 6 : DoS attacks-GoldenEye
 7 : DoS attacks-Hulk
 8 : DoS attacks-SlowHTTPTest
 9 : DoS attacks-Slowloris
10 : FTP-BruteForce
11 : Infilteration
12 : SQL Injection
13 : SSH-Bruteforce


In [37]:
# ============================================================
# Cell 5 - Build Dataset Index
# ============================================================

dataset_records = []

chunk_files = sorted(
    [
        f
        for f in os.listdir(INPUT_DIR)
        if f.startswith("clean_chunk_") and f.endswith(".csv")
    ],
    key=lambda x: int(x.split("_")[-1].split(".")[0])
)

print("=" * 70)
print("Building Dataset Index...")
print("=" * 70)

for chunk_file in tqdm(chunk_files):

    chunk_id = int(chunk_file.split("_")[-1].split(".")[0])

    file_path = os.path.join(INPUT_DIR, chunk_file)

    # Read timestamp column only (fast)
    df = pd.read_csv(
        file_path,
        usecols=[TIMESTAMP_COLUMN]
    )
    
    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )
    
    df = df[
    df[TIMESTAMP_COLUMN].notna()
    ]
    
    df = df[
    df[TIMESTAMP_COLUMN].dt.year >= 2018
    ]

    df = df.sort_values(TIMESTAMP_COLUMN)

    # Count rows for each day
    day_counts = (
        df[TIMESTAMP_COLUMN]
        .dt.date
        .value_counts()
        .sort_index()
    )

    for day, rows in day_counts.items():

        dataset_records.append({

            "Chunk": chunk_id,

            "File": chunk_file,

            "Date": str(day),

            "Rows": int(rows)

        })

    del df
    gc.collect()

# ============================================================
# Dataset Index
# ============================================================

dataset_index = pd.DataFrame(dataset_records)

dataset_index = dataset_index.sort_values(
    ["Chunk", "Date"]
).reset_index(drop=True)

# ============================================================
# Dataset Summary
# ============================================================

total_chunks = dataset_index["Chunk"].nunique()

total_days = len(dataset_index)

total_rows = dataset_index["Rows"].sum()

avg_rows = int(total_rows / total_days)

print()
print("=" * 70)
print("Dataset Summary")
print("=" * 70)

print(f"Total Chunks        : {total_chunks}")
print(f"Total Days          : {total_days}")
print(f"Total Rows          : {total_rows:,}")
print(f"Average Rows / Day  : {avg_rows:,}")

print()
print(
    f"Date Range          : "
    f"{dataset_index['Date'].min()}  →  {dataset_index['Date'].max()}"
)

print("=" * 70)

print()
print(dataset_index.head(10))

Building Dataset Index...


  0%|          | 0/32 [00:00<?, ?it/s]


Dataset Summary
Total Chunks        : 32
Total Days          : 41
Total Rows          : 15,730,957
Average Rows / Day  : 383,681

Date Range          : 2018-02-14  →  2018-03-02

   Chunk               File        Date    Rows
0      1  clean_chunk_1.csv  2018-03-02  498028
1      2  clean_chunk_2.csv  2018-02-16  451425
2      2  clean_chunk_2.csv  2018-03-02   48575
3      3  clean_chunk_3.csv  2018-02-16  500000
4      4  clean_chunk_4.csv  2018-02-16   97149
5      4  clean_chunk_4.csv  2018-02-23  402850
6      5  clean_chunk_5.csv  2018-02-23  500000
7      6  clean_chunk_6.csv  2018-02-20  354275
8      6  clean_chunk_6.csv  2018-02-23  145725
9      7  clean_chunk_7.csv  2018-02-20  500000


In [38]:
# ============================================================
# Cell 6 - Streaming Data Pipeline
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view

# ============================================================
# Create Sliding Window Sequences
# ============================================================

def create_sequences(X, y, window_size):

    """
    Convert a feature matrix into overlapping sequences.

    Parameters
    ----------
    X : ndarray
        Shape (N, num_features)

    y : ndarray
        Shape (N,)

    Returns
    -------
    X_seq : ndarray
        Shape (N-window+1, window_size, num_features)

    y_seq : ndarray
        Shape (N-window+1,)
    """

    if len(X) < window_size:

        return None, None

    X_seq = sliding_window_view(
        X,
        window_shape=window_size,
        axis=0
    )

    # (samples, features, window) -> (samples, window, features)
    X_seq = np.transpose(X_seq, (0, 2, 1))

    y_seq = y[window_size - 1:]

    return (
        X_seq.astype(np.float32),
        y_seq.astype(np.int64)
    )


# ============================================================
# Preprocess One Day
# ============================================================

def preprocess_day(day_df):

    day_df = day_df.sort_values(
        TIMESTAMP_COLUMN
    ).reset_index(drop=True)

    X = day_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

    # -----------------------------
    # Check raw features
    # -----------------------------
    if not np.isfinite(X).all():

        print("=" * 60)
        print("Invalid values BEFORE scaling")
        print("NaN :", np.isnan(X).sum())
        print("Inf :", np.isinf(X).sum())

        raise ValueError("Raw features contain NaN/Inf")

    X = scaler.transform(X)

    # -----------------------------
    # Check scaled features
    # -----------------------------
    if not np.isfinite(X).all():

        print("=" * 60)
        print("Invalid values AFTER scaling")
        print("NaN :", np.isnan(X).sum())
        print("Inf :", np.isinf(X).sum())

        raise ValueError("Scaled features contain NaN/Inf")

    y = label_encoder.transform(
        day_df[LABEL_COLUMN]
    )

    return X, y


# ============================================================
# Build DataLoader
# ============================================================

def create_loader(X, y, shuffle=False):

    dataset = TensorDataset(

        torch.from_numpy(X),

        torch.from_numpy(y)

    )

    return DataLoader(

        dataset,

        batch_size=DATA_CONFIG["batch_size"],

        shuffle=shuffle,

        num_workers=DATA_CONFIG["num_workers"],

        pin_memory=torch.cuda.is_available(),

        persistent_workers=(
            DATA_CONFIG["num_workers"] > 0
        )

    )


# ============================================================
# Prepare Train / Validation / Test Loaders
# ============================================================

def prepare_day_loaders(day_df):

    """
    Returns

    {
        "train": loader,
        "validation": loader,
        "test": loader
    }

    """

    X, y = preprocess_day(day_df)

    n = len(X)

    train_end = int(0.70 * n)

    valid_end = int(0.85 * n)

    splits = {

        "train": (

            X[:train_end],

            y[:train_end]

        ),

        "validation": (

            X[train_end:valid_end],

            y[train_end:valid_end]

        ),

        "test": (

            X[valid_end:],

            y[valid_end:]

        )

    }

    loaders = {}

    for split_name, (X_split, y_split) in splits.items():

        X_seq, y_seq = create_sequences(

            X_split,

            y_split,

            DATA_CONFIG["window_size"]

        )

        if X_seq is None:

            loaders[split_name] = None

            continue

        loaders[split_name] = create_loader(

            X_seq,

            y_seq,

            shuffle=(split_name == "train")

        )

    del X
    del y
    gc.collect()

    return loaders


# ============================================================
# Sanity Check
# ============================================================

print("=" * 70)
print("Testing Data Pipeline...")
print("=" * 70)

sample = dataset_index.iloc[0]

sample_df = pd.read_csv(
    os.path.join(
        INPUT_DIR,
        sample["File"]
    )
)

sample_df[TIMESTAMP_COLUMN] = pd.to_datetime(
    sample_df[TIMESTAMP_COLUMN],
    errors="coerce"
)

sample_df = sample_df[
    sample_df[TIMESTAMP_COLUMN].dt.date.astype(str)
    == sample["Date"]
]

loaders = prepare_day_loaders(sample_df)

print(loaders.keys())

print()

for split_name, loader in loaders.items():

    if loader is None:
        continue

    X_batch, y_batch = next(iter(loader))

    print("=" * 60)
    print(split_name)

    print("X :", X_batch.shape)

    print("y :", y_batch.shape)

del sample_df
gc.collect()

Testing Data Pipeline...
dict_keys(['train', 'validation', 'test'])

train
X : torch.Size([256, 20, 43])
y : torch.Size([256])
validation
X : torch.Size([256, 20, 43])
y : torch.Size([256])
test
X : torch.Size([256, 20, 43])
y : torch.Size([256])


0

In [57]:
# ============================================================
# Cell 7 - Transformer IDS Model
# ============================================================

class TransformerIDS(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Input Projection
        # ----------------------------------------------------

        self.input_projection = nn.Linear(

            MODEL_CONFIG["input_dim"],

            MODEL_CONFIG["d_model"]

        )

        # ----------------------------------------------------
        # Transformer Encoder
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=MODEL_CONFIG["d_model"],

            nhead=MODEL_CONFIG["nhead"],

            dim_feedforward=MODEL_CONFIG["dim_feedforward"],

            dropout=MODEL_CONFIG["dropout"],

            batch_first=True,

            activation="gelu"

        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=MODEL_CONFIG["num_layers"]

        )

        # ----------------------------------------------------
        # Attention Pooling
        # ----------------------------------------------------

        self.attention = nn.Sequential(

            nn.Linear(

                MODEL_CONFIG["d_model"],

                MODEL_CONFIG["attention_hidden"]

            ),

            nn.Tanh(),

            nn.Linear(

                MODEL_CONFIG["attention_hidden"],

                1

            )

        )

        # ----------------------------------------------------
        # Classification Head
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.LayerNorm(

                MODEL_CONFIG["d_model"]

            ),

            nn.Linear(

                MODEL_CONFIG["d_model"],

                MODEL_CONFIG["d_model"]

            ),

            nn.GELU(),

            nn.Dropout(

                MODEL_CONFIG["dropout"]

            ),

            nn.Linear(

                MODEL_CONFIG["d_model"],

                NUM_CLASSES

            )

        )

    # ========================================================
    # Forward
    # ========================================================

    def forward(self, x):
        x = self.input_projection(x)
        # if not torch.isfinite(x).all():
        #     print("NaN after input_projection")
        #     return x
        
        # print("Input to Transformer")
        # print("Shape :", x.shape)
        # print("Min   :", x.min().item())
        # print("Max   :", x.max().item())
        # print("Mean  :", x.mean().item())
        # print("Std   :", x.std().item())

        x = self.transformer(x)

        # if not torch.isfinite(x).all():
        #     print("NaN after transformer")
        #     return x
            
        attention_scores = self.attention(x)
        
        # if not torch.isfinite(attention_scores).all():
        #     print("NaN after attention scores")
        #     return attention_scores
            
        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )
        # if not torch.isfinite(attention_weights).all():
        #     print("NaN after softmax")
        #     return attention_weights
            
        context = torch.sum(
            attention_weights * x,
            dim=1
        )
        
        
            
        output = self.classifier(context)
        # if not torch.isfinite(output).all():
        #     print("NaN after classifier")
        return output

# ============================================================
# Initialize Model
# ============================================================

model = TransformerIDS().to(DEVICE)

print(model)

print()

total_params = sum(

    p.numel()

    for p in model.parameters()

)

trainable_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print("=" * 65)

print(f"Total Parameters     : {total_params:,}")

print(f"Trainable Parameters : {trainable_params:,}")

print("=" * 65)

TransformerIDS(
  (input_projection): Linear(in_features=43, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0

In [58]:
# ============================================================
# Cell 8 - Training Utilities
# ============================================================

from collections import Counter

# ============================================================
# Compute Class Distribution
# ============================================================

print("=" * 70)
print("Computing Class Distribution...")
print("=" * 70)

class_counter = Counter()

for _, row in tqdm(dataset_index.iterrows(), total=len(dataset_index)):

    file_path = os.path.join(
        INPUT_DIR,
        row["File"]
    )

    df = pd.read_csv(
        file_path,
        usecols=[TIMESTAMP_COLUMN, LABEL_COLUMN]
    )

    df[TIMESTAMP_COLUMN] = pd.to_datetime(
        df[TIMESTAMP_COLUMN],
        errors="coerce"
    )

    df = df[
        df[TIMESTAMP_COLUMN].notna()
    ]

    df = df[
        df[TIMESTAMP_COLUMN].dt.date.astype(str)
        == row["Date"]
    ]

    class_counter.update(df[LABEL_COLUMN])

    del df
    gc.collect()

# ============================================================
# Class Counts
# ============================================================

class_counts = np.array(
    [
        class_counter[class_name]
        for class_name in CLASS_NAMES
    ],
    dtype=np.float32
)

# ============================================================
# Square-Root Inverse Frequency Weights
# ============================================================

class_weights = np.sqrt(
    class_counts.max() / class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print()

print("=" * 70)
print("Class Weights")
print("=" * 70)

for cls, w in zip(CLASS_NAMES, class_weights.cpu().numpy()):

    print(f"{cls:<30} {w:.2f}")

print("=" * 70)

# ============================================================
# Loss Function
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=TRAIN_CONFIG["learning_rate"],

    weight_decay=TRAIN_CONFIG["weight_decay"]

)

# ============================================================
# Learning Rate Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=1

)

# ============================================================
# Automatic Mixed Precision
# ============================================================

scaler_amp = GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

# ============================================================
# Training History
# ============================================================

history = {

    "train_loss": [],

    "train_accuracy": [],

    "validation_loss": [],

    "validation_accuracy": []

}

# ============================================================
# Early Stopping
# ============================================================

best_val_loss = float("inf")

best_model_state = None

early_stop_counter = 0

# ============================================================
# Summary
# ============================================================

print()

print("=" * 70)

print("Training Configuration Ready")

print("=" * 70)

print("Criterion :", criterion)

print("Optimizer :", optimizer.__class__.__name__)

print("Scheduler :", scheduler.__class__.__name__)

print("AMP Enabled :", torch.cuda.is_available())

print("=" * 70)

Computing Class Distribution...


  0%|          | 0/41 [00:00<?, ?it/s]


Class Weights
Benign                         1.00
Brute Force -Web               147.37
Brute Force -XSS               240.19
DDOS attack-HOIC               4.40
DDOS attack-LOIC-UDP           87.58
DDoS attacks-LOIC-HTTP         4.80
DoS attacks-GoldenEye          17.88
DoS attacks-Hulk               5.36
DoS attacks-SlowHTTPTest       9.74
DoS attacks-Slowloris          34.75
FTP-BruteForce                 8.28
Infilteration                  9.05
SQL Injection                  390.53
SSH-Bruteforce                 8.41

Training Configuration Ready
Criterion : CrossEntropyLoss()
Optimizer : AdamW
Scheduler : ReduceLROnPlateau
AMP Enabled : True


In [60]:
# ============================================================
# Training State
# ============================================================

history = {

    "train_loss": [],
    "train_accuracy": [],

    "validation_loss": [],
    "validation_accuracy": []

}

best_val_loss = float("inf")

best_model_state = None

early_stop_counter = 0

In [63]:
# ============================================================
# Cell 9 - Model Training
# Part 1 : Helper Functions
# ============================================================

import copy

# ============================================================
# Train One Epoch
# ============================================================

def train_one_epoch(loader):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    for X, y in loader:

        X = X.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            enabled=False
        ):
            # if not torch.isfinite(X).all():
            #     print("=" * 60)
            #     print("Invalid input detected!")
            #     print("NaN :", torch.isnan(X).sum().item())
            #     print("Inf :", torch.isinf(X).sum().item())
            #     print("Min :", torch.nanmin(X).item())
            #     print("Max :", torch.nanmax(X).item())
            #     return float("nan"), 0.0

            outputs = model(X)

            loss = criterion(outputs, y)
            # if not torch.isfinite(loss):
            #     print("=" * 60)
            #     print("Invalid loss detected")
            #     print("Loss:", loss)
            #     print("Min output:", outputs.min().item())
            #     print("Max output:", outputs.max().item())
            #     print("Labels:", torch.unique(y))
            #     return float("nan"), 0.0

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler_amp.step(optimizer)

        scaler_amp.update()

        running_loss += loss.item() * X.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == y).sum().item()

        total += y.size(0)

    epoch_loss = running_loss / total

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


# ============================================================
# Validation
# ============================================================

def evaluate(loader):

    model.eval()

    running_loss = 0.0

    correct = 0

    total = 0

    with torch.no_grad():

        for X, y in loader:

            X = X.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast(
                device_type="cuda",
                enabled=False
            ):

                outputs = model(X)

                loss = criterion(outputs, y)

            running_loss += loss.item() * X.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == y).sum().item()

            total += y.size(0)

    epoch_loss = running_loss / total

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy


# ============================================================
# Epoch Training
# ============================================================

print("=" * 70)
print("Starting Training")
print("=" * 70)

for epoch in range(TRAIN_CONFIG["epochs"]):

    print()

    print("=" * 70)
    print(
        f"Epoch {epoch + 1}/{TRAIN_CONFIG['epochs']}"
    )
    print("=" * 70)

    train_loss_sum = 0.0
    train_acc_sum = 0.0
    train_days = 0

    val_loss_sum = 0.0
    val_acc_sum = 0.0
    val_days = 0

    # ========================================================
    # Iterate Through Dataset
    # ========================================================

    for _, row in dataset_index.iterrows():

        file_path = os.path.join(
            INPUT_DIR,
            row["File"]
        )

        # ----------------------------------------------------
        # Load One Chunk
        # ----------------------------------------------------

        day_df = pd.read_csv(file_path)

        day_df[TIMESTAMP_COLUMN] = pd.to_datetime(
            day_df[TIMESTAMP_COLUMN],
            errors="coerce"
        )

        day_df = day_df[
            day_df[TIMESTAMP_COLUMN].notna()
        ]

        day_df = day_df[
            day_df[TIMESTAMP_COLUMN].dt.date.astype(str)
            == row["Date"]
        ]

        if len(day_df) < DATA_CONFIG["window_size"]:

            del day_df
            gc.collect()
            continue

        # ----------------------------------------------------
        # Create Loaders
        # ----------------------------------------------------

        loaders = prepare_day_loaders(day_df)

        del day_df
        gc.collect()

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        if loaders["train"] is not None:

            loss, acc = train_one_epoch(
                loaders["train"]
            )

            train_loss_sum += loss
            train_acc_sum += acc
            train_days += 1

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        if loaders["validation"] is not None:

            loss, acc = evaluate(
                loaders["validation"]
            )

            val_loss_sum += loss
            val_acc_sum += acc
            val_days += 1

        # ----------------------------------------------------
        # Free Memory
        # ----------------------------------------------------

        del loaders

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

    # ========================================================
    # Epoch Metrics
    # ========================================================

    train_loss = train_loss_sum / max(train_days, 1)

    train_accuracy = train_acc_sum / max(train_days, 1)

    validation_loss = val_loss_sum / max(val_days, 1)

    validation_accuracy = val_acc_sum / max(val_days, 1)

    print()

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Train Accuracy : {train_accuracy:.4f}"
    )

    print()

    print(
        f"Validation Loss : {validation_loss:.4f}"
    )

    print(
        f"Validation Accuracy : {validation_accuracy:.4f}"
    )

    # ========================================================
    # Learning Rate Scheduler
    # ========================================================

    scheduler.step(validation_loss)

    # ========================================================
    # Save History
    # ========================================================

    history["train_loss"].append(train_loss)

    history["train_accuracy"].append(train_accuracy)

    history["validation_loss"].append(validation_loss)

    history["validation_accuracy"].append(validation_accuracy)

    # ========================================================
    # Best Model
    # ========================================================

    if validation_loss < best_val_loss:

        best_val_loss = validation_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        early_stop_counter = 0

        print()

        print("✓ Best model updated.")

    else:

        early_stop_counter += 1

        print()

        print(
            f"No improvement "
            f"({early_stop_counter}/"
            f"{TRAIN_CONFIG['patience']})"
        )

    # ========================================================
    # Early Stopping
    # ========================================================

    if early_stop_counter >= TRAIN_CONFIG["patience"]:

        print()

        print("=" * 70)

        print("Early stopping triggered.")

        print("=" * 70)

        break

# ============================================================
# Restore Best Model
# ============================================================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

# ============================================================
# Training Complete
# ============================================================

print()

print("=" * 70)

print("Training Complete")

print("=" * 70)

print(f"Best Validation Loss : {best_val_loss:.4f}")

print("=" * 70)

Starting Training

Epoch 1/15

Train Loss : 0.1878
Train Accuracy : 0.9163

Validation Loss : 0.4407
Validation Accuracy : 0.9205

✓ Best model updated.

Epoch 2/15

Train Loss : 0.2204
Train Accuracy : 0.9151

Validation Loss : 0.4218
Validation Accuracy : 0.9074

✓ Best model updated.

Epoch 3/15

Train Loss : 0.2059
Train Accuracy : 0.9130

Validation Loss : 0.4575
Validation Accuracy : 0.9229

No improvement (1/5)

Epoch 4/15

Train Loss : 0.1971
Train Accuracy : 0.9183

Validation Loss : 0.5390
Validation Accuracy : 0.9164

No improvement (2/5)

Epoch 5/15

Train Loss : 0.2829
Train Accuracy : 0.9034

Validation Loss : 0.4172
Validation Accuracy : 0.9021

✓ Best model updated.

Epoch 6/15

Train Loss : 0.2550
Train Accuracy : 0.9154

Validation Loss : 0.5726
Validation Accuracy : 0.9033

No improvement (1/5)

Epoch 7/15

Train Loss : 0.2584
Train Accuracy : 0.9194

Validation Loss : 0.5302
Validation Accuracy : 0.9110

No improvement (2/5)

Epoch 8/15

Train Loss : 0.3095
Train Ac

In [73]:
torch.save(model.state_dict(), "best_transformer_model.pth")

joblib.dump(history, "training_history.pkl")

['training_history.pkl']

****TEST EVALUATION**** 

In [65]:
# ============================================================
# Cell 1 - Imports
# ============================================================

import os
import gc
import pickle
import joblib
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix,

    roc_auc_score,

    roc_curve

)

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("=" * 65)
print("Libraries Loaded Successfully")
print("=" * 65)

Libraries Loaded Successfully


In [77]:
# ============================================================
# Cell 2 - Configuration
# ============================================================

DATA_PATHS = {

    "htre_dataset": "/path/to/dest/CIC_IDS2018_HTRE_Final",

    "artifacts": "/path/to/dest/Transformer_Pipeline"

}

MODEL_PATH = "/kaggle/working/best_transformer_model.pth"

ARTIFACT_PATHS = {

    "feature_columns":

        os.path.join(
            DATA_PATHS["artifacts"],
            "feature_columns.pkl"
        ),

    "label_encoder":

        os.path.join(
            DATA_PATHS["artifacts"],
            "label_encoder.pkl"
        ),

    "scaler":

        os.path.join(
            DATA_PATHS["artifacts"],
            "standard_scaler.pkl"
        )

}

MODEL_CONFIG = {

    "d_model":128,

    "nhead":4,

    "num_layers":2,

    "dim_feedforward":256,

    "dropout":0.2,

    "attention_hidden":64

}

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("="*65)
print("Device :", DEVICE)
print("="*65)

Device : cuda


In [78]:
# ============================================================
# Cell 3 - Load Artifacts
# ============================================================

# -----------------------------
# Load Feature Columns
# -----------------------------

feature_columns = joblib.load(
    ARTIFACT_PATHS["feature_columns"]
)

# -----------------------------
# Load Standard Scaler
# -----------------------------

scaler = joblib.load(
    ARTIFACT_PATHS["scaler"]
)

# -----------------------------
# Load Label Encoder
# -----------------------------

label_encoder = joblib.load(
    ARTIFACT_PATHS["label_encoder"]
)

# -----------------------------
# Update Model Configuration
# -----------------------------

MODEL_CONFIG["input_dim"] = len(feature_columns)

NUM_CLASSES = len(label_encoder.classes_)

# -----------------------------
# Dataset Constants
# -----------------------------

TIMESTAMP_COLUMN = "Timestamp"

LABEL_COLUMN = "Label"

print("=" * 65)
print("Artifacts Loaded Successfully")
print("=" * 65)

print(f"Features : {len(feature_columns)}")
print(f"Classes  : {NUM_CLASSES}")

print()
print("Feature Preview")
print(feature_columns[:5], "...")

print()
print("Classes")

for i, cls in enumerate(label_encoder.classes_):

    print(f"{i:2d} : {cls}")

print("=" * 65)

Artifacts Loaded Successfully
Features : 43
Classes  : 14

Feature Preview
['Init Fwd Win Byts', 'Fwd Seg Size Min', 'Fwd Header Len', 'Dst Port', 'Flow Duration'] ...

Classes
 0 : Benign
 1 : Brute Force -Web
 2 : Brute Force -XSS
 3 : DDOS attack-HOIC
 4 : DDOS attack-LOIC-UDP
 5 : DDoS attacks-LOIC-HTTP
 6 : DoS attacks-GoldenEye
 7 : DoS attacks-Hulk
 8 : DoS attacks-SlowHTTPTest
 9 : DoS attacks-Slowloris
10 : FTP-BruteForce
11 : Infilteration
12 : SQL Injection
13 : SSH-Bruteforce


In [79]:
# ============================================================
# Cell 4 - Transformer Model
# ============================================================

class TransformerIDS(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Input Projection
        # ----------------------------------------------------

        self.input_projection = nn.Linear(
            MODEL_CONFIG["input_dim"],
            MODEL_CONFIG["d_model"]
        )

        # ----------------------------------------------------
        # Transformer Encoder
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=MODEL_CONFIG["d_model"],

            nhead=MODEL_CONFIG["nhead"],

            dim_feedforward=MODEL_CONFIG["dim_feedforward"],

            dropout=MODEL_CONFIG["dropout"],

            batch_first=True,

            activation="gelu"

        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=MODEL_CONFIG["num_layers"]

        )

        # ----------------------------------------------------
        # Attention Pooling
        # ----------------------------------------------------

        self.attention = nn.Sequential(

            nn.Linear(
                MODEL_CONFIG["d_model"],
                MODEL_CONFIG["attention_hidden"]
            ),

            nn.Tanh(),

            nn.Linear(
                MODEL_CONFIG["attention_hidden"],
                1
            )

        )

        # ----------------------------------------------------
        # Classification Head
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                MODEL_CONFIG["d_model"]
            ),

            nn.Linear(
                MODEL_CONFIG["d_model"],
                MODEL_CONFIG["d_model"]
            ),

            nn.GELU(),

            nn.Dropout(
                MODEL_CONFIG["dropout"]
            ),

            nn.Linear(
                MODEL_CONFIG["d_model"],
                NUM_CLASSES
            )

        )

    def forward(self, x):

        # Input Projection
        x = self.input_projection(x)

        # Transformer Encoder
        x = self.transformer(x)

        # Attention Pooling
        attention_scores = self.attention(x)

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        context = torch.sum(
            attention_weights * x,
            dim=1
        )

        # Classification
        output = self.classifier(context)

        return output


# ============================================================
# Create Model
# ============================================================

model = TransformerIDS().to(DEVICE)

print(model)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 65)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print("=" * 65)

TransformerIDS(
  (input_projection): Linear(in_features=43, out_features=128, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0

In [81]:
# ============================================================
# Cell 5 - Load Trained Model
# ============================================================

print("=" * 65)
print("Loading Trained Transformer...")
print("=" * 65)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)
model.load_state_dict(checkpoint)

model.eval()

print("✓ Model weights loaded successfully.")
print("✓ Model switched to evaluation mode.")

print("=" * 65)

Loading Trained Transformer...
✓ Model weights loaded successfully.
✓ Model switched to evaluation mode.


In [83]:
# ============================================================
# Cell 6 - Streaming Data Pipeline
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view

# ============================================================
# Create Sliding Window Sequences
# ============================================================

def create_sequences(X, y, window_size):

    """
    Convert a feature matrix into overlapping sequences.

    Parameters
    ----------
    X : ndarray
        Shape (N, num_features)

    y : ndarray
        Shape (N,)

    Returns
    -------
    X_seq : ndarray
        Shape (N-window+1, window_size, num_features)

    y_seq : ndarray
        Shape (N-window+1,)
    """

    if len(X) < window_size:

        return None, None

    X_seq = sliding_window_view(
        X,
        window_shape=window_size,
        axis=0
    )

    # (samples, features, window) -> (samples, window, features)
    X_seq = np.transpose(X_seq, (0, 2, 1))

    y_seq = y[window_size - 1:]

    return (
        X_seq.astype(np.float32),
        y_seq.astype(np.int64)
    )


# ============================================================
# Preprocess One Day
# ============================================================

def preprocess_day(day_df):

    day_df = day_df.sort_values(
        TIMESTAMP_COLUMN
    ).reset_index(drop=True)

    X = day_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

    # -----------------------------
    # Check raw features
    # -----------------------------
    if not np.isfinite(X).all():

        print("=" * 60)
        print("Invalid values BEFORE scaling")
        print("NaN :", np.isnan(X).sum())
        print("Inf :", np.isinf(X).sum())

        raise ValueError("Raw features contain NaN/Inf")

    X = scaler.transform(X)

    # -----------------------------
    # Check scaled features
    # -----------------------------
    if not np.isfinite(X).all():

        print("=" * 60)
        print("Invalid values AFTER scaling")
        print("NaN :", np.isnan(X).sum())
        print("Inf :", np.isinf(X).sum())

        raise ValueError("Scaled features contain NaN/Inf")

    y = label_encoder.transform(
        day_df[LABEL_COLUMN]
    )

    return X, y


# ============================================================
# Build DataLoader
# ============================================================

def create_loader(X, y, shuffle=False):

    dataset = TensorDataset(

        torch.from_numpy(X),

        torch.from_numpy(y)

    )

    return DataLoader(

        dataset,

        batch_size=DATA_CONFIG["batch_size"],

        shuffle=shuffle,

        num_workers=DATA_CONFIG["num_workers"],

        pin_memory=torch.cuda.is_available(),

        persistent_workers=(
            DATA_CONFIG["num_workers"] > 0
        )

    )


# ============================================================
# Prepare Train / Validation / Test Loaders
# ============================================================
def prepare_test_loader(day_df):

    X, y = preprocess_day(day_df)

    n = len(X)

    test_start = int(0.85 * n)

    X_test = X[test_start:]
    y_test = y[test_start:]

    X_seq, y_seq = create_sequences(
        X_test,
        y_test,
        DATA_CONFIG["window_size"]
    )

    if X_seq is None:
        return None

    loader = create_loader(
        X_seq,
        y_seq,
        shuffle=False
    )

    del X
    del y
    gc.collect()

    return loader

# ============================================================
# Sanity Check
# ============================================================

print("=" * 70)
print("Testing Data Pipeline...")
print("=" * 70)

sample = dataset_index.iloc[0]

sample_df = pd.read_csv(
    os.path.join(
        INPUT_DIR,
        sample["File"]
    )
)

sample_df[TIMESTAMP_COLUMN] = pd.to_datetime(
    sample_df[TIMESTAMP_COLUMN],
    errors="coerce"
)

sample_df = sample_df[
    sample_df[TIMESTAMP_COLUMN].dt.date.astype(str)
    == sample["Date"]
]

test_loader = prepare_test_loader(sample_df)

if test_loader is not None:

    X_batch, y_batch = next(iter(test_loader))

    print("=" * 60)
    print("Test Loader")

    print("X :", X_batch.shape)
    print("y :", y_batch.shape)

del sample_df
gc.collect()

Testing Data Pipeline...
Test Loader
X : torch.Size([256, 20, 43])
y : torch.Size([256])


0

In [84]:
# ============================================================
# Cell 7 - Run Inference on Test Set
# ============================================================

from tqdm.auto import tqdm

print("=" * 70)
print("Running Inference on Test Set...")
print("=" * 70)

# ============================================================
# Storage
# ============================================================

y_true = []
y_pred = []
y_prob = []

# ============================================================
# Inference
# ============================================================

model.eval()

with torch.no_grad():

    for _, row in tqdm(dataset_index.iterrows(),
                       total=len(dataset_index)):

        file_path = os.path.join(
            INPUT_DIR,
            row["File"]
        )

        day_df = pd.read_csv(file_path)

        day_df[TIMESTAMP_COLUMN] = pd.to_datetime(
            day_df[TIMESTAMP_COLUMN],
            errors="coerce"
        )

        day_df = day_df[
            day_df[TIMESTAMP_COLUMN].dt.date.astype(str)
            == row["Date"]
        ]

        test_loader = prepare_test_loader(day_df)

        del day_df
        gc.collect()

        if test_loader is None:
            continue

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(DEVICE)

            outputs = model(X_batch)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_prob.extend(
                probabilities.cpu().numpy()
            )

        del test_loader
        gc.collect()

# ============================================================
# Convert to NumPy
# ============================================================

y_true = np.array(y_true)

y_pred = np.array(y_pred)

y_prob = np.array(y_prob)

print()

print("=" * 70)
print("Inference Complete")
print("=" * 70)

print("Samples Evaluated :", len(y_true))

print("Prediction Shape  :", y_pred.shape)

print("Probability Shape :", y_prob.shape)

print("=" * 70)

Running Inference on Test Set...


  0%|          | 0/41 [00:00<?, ?it/s]


Inference Complete
Samples Evaluated : 2358874
Prediction Shape  : (2358874,)
Probability Shape : (2358874, 14)


In [86]:
# ============================================================
# Cell 8 - Classification Metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("=" * 70)
print("Overall Test Performance")
print("=" * 70)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print()

print("=" * 70)
print("Classification Report")
print("=" * 70)

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
)

print(report)

Overall Test Performance
Accuracy  : 0.2834
Precision : 0.7766
Recall    : 0.2834
F1-Score  : 0.4096

Classification Report
                          precision    recall  f1-score   support

                  Benign     0.9086    0.3254    0.4792   2015889
        Brute Force -Web     0.0000    0.0000    0.0000         3
        Brute Force -XSS     0.0000    0.0000    0.0000         0
        DDOS attack-HOIC     0.0000    0.0000    0.0000     76736
    DDOS attack-LOIC-UDP     0.0000    0.0000    0.0000      1730
  DDoS attacks-LOIC-HTTP     0.0000    0.0000    0.0000     61585
   DoS attacks-GoldenEye     0.0000    0.0000    0.0000         0
        DoS attacks-Hulk     0.0000    0.0000    0.0000     41973
DoS attacks-SlowHTTPTest     0.0000    0.0000    0.0000     82123
   DoS attacks-Slowloris     0.0000    0.0000    0.0000      5907
          FTP-BruteForce     0.0000    0.0000    0.0000     54161
           Infilteration     0.0078    0.6627    0.0154     18767
           SQL In